In [64]:
import pandas as pd
df = pd.read_csv("../data/processed/superstore_cleaned.csv")
df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,product_name,sales,quantity,discount,profit,order_year,order_month,order_month_name,is_outlier,profit_margin
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,November,False,0.1600
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,November,True,0.3000
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,June,False,0.4700
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,October,True,-0.4000
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,October,False,0.1125


In [65]:
df["product_id"] = df["product_name"].astype('category').cat.codes

**generate orders.csv for feature engineering**

In [66]:
orders = df[
    [
        "order_id",
        "order_date",
        "ship_date",
        "customer_id",
        "product_id",
        "quantity",
        "sales",
        "profit",
        "discount",
        "order_year",
        "order_month",
        "is_outlier"
    ]
]


In [67]:
orders.to_csv("../data/processed/orders.csv", index=False)

**generate customer.csv for feature engineering**

In [68]:
customers = ( df.groupby("customer_id").agg(
    customer_name = ("customer_name", "first"),
    segment = ("segment","first"),
    region = ("region","first"),
    state = ("state","first"),
    city = ("city","first"),
    first_order_date = ("order_date", "min"),
    last_order_date = ("order_date", "max"),
    total_orders = ("order_id", "nunique"),
    total_revenue = ("sales","sum"),
    total_profit = ("profit", "sum"),

)
.reset_index()
)

**Customer Lifetime Value (LTV)**

In [69]:
customers["customer_ltv"] = customers["total_revenue"]

In [70]:
customers["customer_type"] = customers["total_orders"].apply(
    lambda x: "repeat" if x > 1 else "new"
)

In [71]:
customers.to_csv("../data/processed/customers.csv",index=False)

**creating product table**

In [72]:
products = (
    df[["product_id","product_name","category","sub_category"]]
    .drop_duplicates()
)

In [73]:
products.to_csv("../data/processed/products.csv", index=False)

**checking and validation**

In [74]:
customers["customer_id"].is_unique

True

In [75]:
products["product_id"].is_unique

True

In [76]:
orders["sales"].sum() == customers["total_revenue"].sum()

np.False_

In [77]:
orders.duplicated().sum()


np.int64(1)

In [78]:
orders["sales"].sum()

np.float64(2297200.8603000003)

In [79]:
customers["total_revenue"].sum()

np.float64(2297200.8603)

In [84]:
orders.duplicated().sum()



np.int64(1)

In [80]:
orders.duplicated().sum()

np.int64(1)

In [81]:
orders[orders.duplicated(keep=False)]


,order_id,order_date,ship_date,customer_id,product_id,quantity,sales,profit,discount,order_year,order_month,is_outlier
3405,US-2014-150119,2014-04-23,2014-04-27,LB-16795,759,2,281.372,-12.0588,0.3,2014,4,False
3406,US-2014-150119,2014-04-23,2014-04-27,LB-16795,759,2,281.372,-12.0588,0.3,2014,4,False


In [85]:
round(orders["sales"].sum(), 2) == round(customers["total_revenue"].sum(), 2)


np.True_